# Notebook 03 — DML nas Tabelas Delta Lake (INSERT / UPDATE / DELETE)

Demonstra as **3 operações DML** (Data Manipulation Language) sobre tabelas Delta Lake no bucket `bronze`, com:

- **INSERT** — Adição de novos registros
- **UPDATE** — Atualização de registros existentes
- **DELETE** — Remoção de registros
- **HISTORY** — Visualização do log de versões
- **TIME TRAVEL** — Consulta a versões anteriores dos dados

Trabalharemos com 3 tabelas: `marca`, `cliente` e `apolice`.

## 1. Importações e SparkSession

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DecimalType
from delta import configure_spark_with_delta_pip, DeltaTable

load_dotenv(find_dotenv())

MINIO_ENDPOINT      = os.getenv('MINIO_ENDPOINT',      'http://localhost:9020')
MINIO_ACCESS_KEY    = os.getenv('MINIO_ACCESS_KEY',    'minioadmin')
MINIO_SECRET_KEY    = os.getenv('MINIO_SECRET_KEY',    'minioadmin')
MINIO_BRONZE_BUCKET = os.getenv('MINIO_BRONZE_BUCKET', 'bronze')

def bronze_path(tabela: str) -> str:
    return f's3a://{MINIO_BRONZE_BUCKET}/{tabela}'


builder = (
    SparkSession.builder
    .appName('DML_Delta_Bronze')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.signing-algorithm', 'S3SignerType')
    .config('spark.jars.packages',
            'io.delta:delta-spark_2.12:3.2.0,'
            'org.apache.hadoop:hadoop-aws:3.3.4,'
            'com.amazonaws:aws-java-sdk-bundle:1.12.262')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '4')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} iniciado.')

---
## 2. Estado inicial das tabelas

Visualizamos o estado atual antes de qualquer DML.

In [ ]:
for tabela in ['marca', 'cliente', 'apolice']:
    df = spark.read.format('delta').load(bronze_path(tabela))
    print(f'\n[{tabela}]  {df.count()} registros:')
    df.show(5, truncate=False)

---
## 3. INSERT — Inserir novos registros

### 3.1 INSERT em `marca` — Novas marcas de veículos

Usamos a API Python do Delta (merge) e também Spark SQL para demonstrar as duas abordagens.

In [ ]:
# --- INSERT via Spark SQL ---
spark.read.format('delta').load(bronze_path('marca')).createOrReplaceTempView('marca')

novas_marcas_sql = """
    SELECT 11 AS id_marca, 'Tesla'  AS nome_marca, 'EUA'     AS pais_origem
    UNION ALL
    SELECT 12,              'BYD',                  'China'
    UNION ALL
    SELECT 13,              'GWM',                  'China'
"""

df_novas_marcas = spark.sql(novas_marcas_sql)

# Append às tabelas Delta
(
    df_novas_marcas.write
    .format('delta')
    .mode('append')
    .save(bronze_path('marca'))
)

df_marca_pos = spark.read.format('delta').load(bronze_path('marca'))
print(f'marca após INSERT: {df_marca_pos.count()} registros')
df_marca_pos.orderBy('id_marca', ascending=False).show(5, truncate=False)

In [ ]:
### 3.2 INSERT em `cliente` — Novo cliente via DeltaTable API (MERGE/INSERT)

from decimal import Decimal

novo_cliente = spark.createDataFrame(
    [(21, 'Valentina Cunha', '222.222.222-21', '2000-05-15', 'valentina.cunha@email.com', 'F')],
    schema=['id_cliente', 'nome', 'cpf', 'data_nascimento', 'email', 'sexo']
)

dt_cliente = DeltaTable.forPath(spark, bronze_path('cliente'))

# MERGE: insere somente se id_cliente não existir
(
    dt_cliente.alias('target')
    .merge(
        novo_cliente.alias('source'),
        'target.id_cliente = source.id_cliente'
    )
    .whenNotMatchedInsertAll()
    .execute()
)

df_cli = spark.read.format('delta').load(bronze_path('cliente'))
print(f'cliente após INSERT: {df_cli.count()} registros')
df_cli.filter(F.col('id_cliente') == 21).show(truncate=False)

---
## 4. UPDATE — Atualizar registros existentes

### 4.1 UPDATE em `marca` — Corrigir país de origem via Spark SQL

In [ ]:
dt_marca = DeltaTable.forPath(spark, bronze_path('marca'))

# UPDATE via Spark SQL
spark.read.format('delta').load(bronze_path('marca')).createOrReplaceTempView('marca_view')
spark.sql("""
    UPDATE delta.`s3a://bronze/marca`
    SET pais_origem = 'Coreia do Sul'
    WHERE nome_marca = 'Hyundai'
""")

print('marca após UPDATE (Hyundai):')
spark.read.format('delta').load(bronze_path('marca')).filter(F.col('nome_marca') == 'Hyundai').show(truncate=False)

In [ ]:
### 4.2 UPDATE em `apolice` — Expirar apólices vencidas via DeltaTable API

from pyspark.sql.functions import lit, current_date

dt_apolice = DeltaTable.forPath(spark, bronze_path('apolice'))

# UPDATE via DeltaTable Python API
dt_apolice.update(
    condition = F.col('data_fim') < F.current_date(),
    set       = {'status': F.lit('Expirada')}
)

df_ap = spark.read.format('delta').load(bronze_path('apolice'))
print('apolice após UPDATE (status Expirada):')
df_ap.groupBy('status').count().show()

In [ ]:
### 4.3 UPDATE em `cliente` — Atualizar email de um cliente via DeltaTable API

dt_cliente = DeltaTable.forPath(spark, bronze_path('cliente'))

dt_cliente.update(
    condition = F.col('id_cliente') == 1,
    set       = {'email': F.lit('ana.souza.novo@email.com')}
)

print('cliente após UPDATE (email Ana Souza):')
spark.read.format('delta').load(bronze_path('cliente')).filter(F.col('id_cliente') == 1).show(truncate=False)

---
## 5. DELETE — Remover registros

### 5.1 DELETE em `marca` — Remover marca via Spark SQL

In [ ]:
print('Antes do DELETE — marcas com id >= 11:')
spark.read.format('delta').load(bronze_path('marca')).filter(F.col('id_marca') >= 11).show(truncate=False)

# DELETE via Spark SQL
spark.sql("""
    DELETE FROM delta.`s3a://bronze/marca`
    WHERE id_marca = 13
""")

print('\nDepois do DELETE (removeu GWM):')
spark.read.format('delta').load(bronze_path('marca')).filter(F.col('id_marca') >= 11).show(truncate=False)

In [ ]:
### 5.2 DELETE em `cliente` — Remover cliente via DeltaTable API

dt_cliente = DeltaTable.forPath(spark, bronze_path('cliente'))

print(f'Antes do DELETE — total clientes: {spark.read.format("delta").load(bronze_path("cliente")).count()}')

# DELETE via DeltaTable Python API
dt_cliente.delete(condition = F.col('id_cliente') == 21)

df_cli_pos = spark.read.format('delta').load(bronze_path('cliente'))
print(f'Depois do DELETE — total clientes: {df_cli_pos.count()}')
print('Verificação (id=21 não deve existir):')
df_cli_pos.filter(F.col('id_cliente') == 21).show()

---
## 6. HISTORY — Histórico de versões do Delta Lake

O Delta Lake registra cada operação no `_delta_log`. Podemos ver o histórico completo de versões.

In [ ]:
for tabela in ['marca', 'cliente', 'apolice']:
    dt = DeltaTable.forPath(spark, bronze_path(tabela))
    print(f"\n{'='*60}")
    print(f'HISTORY — tabela: {tabela}')
    (
        dt.history()
        .select('version', 'timestamp', 'operation', 'operationParameters', 'operationMetrics')
        .show(truncate=False)
    )

---
## 7. TIME TRAVEL — Consultar versão anterior dos dados

O Time Travel é uma das features mais poderosas do Delta Lake: permite ler qualquer versão histórica dos dados.

In [ ]:
# Versão 0 da tabela marca (antes de qualquer DML)
print('marca — versão 0 (estado original após carga inicial):')
df_v0 = (
    spark.read
    .format('delta')
    .option('versionAsOf', 0)
    .load(bronze_path('marca'))
)
df_v0.show(truncate=False)
print(f'Registros na versão 0: {df_v0.count()}')

In [ ]:
# Versão mais recente (após todos os DMLs)
df_atual = spark.read.format('delta').load(bronze_path('marca'))
print('marca — versão atual (após todos os DMLs):')
df_atual.show(truncate=False)
print(f'Registros na versão atual: {df_atual.count()}')

In [ ]:
# Comparação entre versão 0 e atual de 'cliente'
print('cliente — versão 0 vs atual:')
df_cli_v0    = spark.read.format('delta').option('versionAsOf', 0).load(bronze_path('cliente'))
df_cli_atual = spark.read.format('delta').load(bronze_path('cliente'))

print(f'  Versão 0   : {df_cli_v0.count()} registros')
print(f'  Atual      : {df_cli_atual.count()} registros')

print('\nEmail do cliente id=1 na versão 0:')
df_cli_v0.filter(F.col('id_cliente') == 1).select('id_cliente', 'nome', 'email').show(truncate=False)

print('Email do cliente id=1 na versão atual:')
df_cli_atual.filter(F.col('id_cliente') == 1).select('id_cliente', 'nome', 'email').show(truncate=False)

---
## 8. Resumo das Operações DML realizadas

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║              RESUMO DAS OPERAÇÕES DML — DELTA LAKE (bronze)         ║
╠══════════╦════════════╦════════════════════════════════════════════╣
║ Operação ║  Tabela    ║  Descrição                                 ║
╠══════════╬════════════╬════════════════════════════════════════════╣
║ INSERT   ║ marca      ║ Novas marcas: Tesla, BYD, GWM (Spark SQL)  ║
║ INSERT   ║ cliente    ║ Novo cliente id=21 (DeltaTable MERGE API)  ║
║ UPDATE   ║ marca      ║ Hyundai → pais_origem corrigido (SQL)      ║
║ UPDATE   ║ apolice    ║ Status = 'Expirada' p/ vencidas (API)      ║
║ UPDATE   ║ cliente    ║ Email da Ana Souza atualizado (API)        ║
║ DELETE   ║ marca      ║ Removeu GWM (id=13) via Spark SQL          ║
║ DELETE   ║ cliente    ║ Removeu cliente id=21 via DeltaTable API   ║
╚══════════╩════════════╩════════════════════════════════════════════╝

Cada operação gerou uma nova versão no _delta_log.
Time Travel permite consultar qualquer versão anterior.
""")

In [ ]:
spark.stop()
print('SparkSession encerrada.')